In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.options import Options
import re
import time
import json
import pandas as pd
import os
import unicodedata
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading
import numpy as np
from typing import Dict, Any, Optional, Iterable, Tuple

MAX_WORKERS = 3
BASE_PATH = r'C:\Users\Dinh Binh An\OneDrive\Dai_hoc\nhap_mon_python\final_project\project_root\Data\crawled_urls\csv_urls'
OUT_PATH = r'C:\Users\Dinh Binh An\OneDrive\Dai_hoc\nhap_mon_python\final_project\project_root\Data\crawled_json_info'


In [2]:
# Class tiện tích
class Utils:
    # Chuyển các kiểu numpy sang kiểu python tương ứng để json.dump không lỗi.
    @staticmethod
    def convert_np(obj: Any) -> Any:
        if isinstance(obj, dict):
            return {k: Utils.convert_np(v) for k, v in obj.items()}
        elif isinstance(obj, list):
            return [Utils.convert_np(x) for x in obj]
        elif isinstance(obj, (np.integer, np.int64, np.int32)):
            return int(obj)
        elif isinstance(obj, (np.floating, np.float64, np.float32)):
            return float(obj)
        else:
            return obj



In [3]:
# Lớp trích xuất thông tin Listing
class Flattener:

    @staticmethod
    def flatten_listing(data: Dict[str, Any]) -> Dict[str, Any]:
        # Kiểm tra đầu vào bắt buộc phải là dict JSON
        if not isinstance(data, dict):
            raise ValueError("Input phải là một dict JSON.")
        
        flat: Dict[str, Any] = {}

        if 'address' in data:
            flat['address'] = data['address']

        # Lấy phần "specs" – chứa các thông tin chi tiết của listing 
        specs = data.get('specs', {})

        # Chỉ xử lý khi specs là dict
        if isinstance(specs, dict):
            for key, val in specs.items():

                # TH1: value không phải dict → lấy trực tiếp
                if not isinstance(val, dict):
                    flat[key] = val
                    continue

                # TH2: value là dict → thường chứa field "raw"
                raw_val = val.get('raw')

                # Nếu raw có giá trị hợp lệ (không None, không rỗng)
                if raw_val not in [None, '', [], {}]:
                    flat[key] = raw_val
        return flat


In [4]:
class ChromeScraper:

    def __init__(self, headless: bool = False, window_size: Tuple[int, int] = (1920, 1080)):
        self.headless = headless
        self.window_size = window_size

    def _make_options(self, headless: Optional[bool] = None) -> Options:

        is_headless = self.headless if headless is None else headless
        # Cài đặt trình duyệt chrome

        chrome_options = Options() # Tạo đối tượng cấu hình cho Chrome
        if headless:
            chrome_options.add_argument("--headless=new") # chạy trình duyệt ẩn

        chrome_options.add_argument("--disable-gpu") # Tắt tăng tốc GPU để giảm lỗi khi chạy trên server
        chrome_options.add_argument("--no-sandbox") # Tắt sandbox để tránh lỗi khi chạy trong môi trường không root
        chrome_options.add_argument("--blink-settings=imagesEnabled=false")  # Không tải ảnh
        chrome_options.add_argument("--window-size=1920,1080")  # thêm kích thước cửa sổ
        chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/142.0.0.0 Safari/537.36")

        return chrome_options
    

    def scrape_specs(self, url: str, timeout: int = 30, headless: Optional[bool] = None) -> Dict[str, Any]:

        chrome_options = self._make_options(headless=headless)
        service = Service(ChromeDriverManager().install())
        driver = webdriver.Chrome(service=service, options=chrome_options)

        try:
            # Mở URL chỉ định
            driver.get(url)
            print(f"➡️ Đang mở trang: {url}")
            # Chờ trang tải xong một chút để tránh lỗi khi truy cập sớm
            time.sleep(2)
            
            try:
                # Giảm mức zoom của trang để hiển thị nhiều nội dung hơn
                driver.execute_script("document.body.style.zoom='50%'")
            except Exception:
                pass
            
            # Khởi tạo webdriverwait
            wait = WebDriverWait(driver, timeout)
            print("⏳ Đang chờ tải thông tin cơ bản...")
            wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "h1.re__pr-title.js__pr-title")))

            # Lấy địa chỉ
            try:
                address_el = driver.find_element(By.CSS_SELECTOR, "span.re__pr-short-description.js__pr-address")
                address = address_el.text.strip()
            except Exception:
                address = None

            # Chờ phần chứa thông số kỹ thuật
            print("⏳ Đang chờ phần thông tin chi tiết...")
            container_selector = "div.re__pr-specs-content-item"

            # Đợi toàn bộ các phần tử chứa thông số kỹ thuật xuất hiện
            wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, container_selector)))
            time.sleep(1)

            # Lấy toàn bộ các dòng thông số
            items = driver.find_elements(By.CSS_SELECTOR, container_selector)
            print(f"✅ Tìm thấy {len(items)} thông số kỹ thuật.")

            # Dictionary chứa thông số kỹ thuật
            specs: Dict[str, Dict[str, str]] = {}
            for it in items:
                try:
                    # Mỗi dòng có hai phần: tên và giá trị
                    title_el = it.find_element(By.CSS_SELECTOR, "span[class*='title']")
                    value_el = it.find_element(By.CSS_SELECTOR, "span[class*='value']")

                    # Lấy nội dung và loại bỏ khoảng trắng thừa
                    t = title_el.text.strip()
                    v = value_el.text.strip()

                    # Lưu vào dictionary
                    specs[t] = {"raw": v}
                except Exception:
                    # Nếu dòng nào bị lỗi (thiếu title hoặc value) thì bỏ qua
                    continue

            return {"address": address, "specs": specs}
        
        except Exception as e:
            # Xử lý lỗi toàn cục
            print("❌ Lỗi khi tải hoặc trích xuất:", e)
            print("HTML hiện tại (rút gọn):")
            print(driver.page_source[:1500])
            return {}
        
        finally:
            # Đóng trình duyệt
            driver.quit()


In [5]:
class CrawlerWorker:
    # Danh sách tên quận 
    DISTRICT_NAMES = [
        '1','2','3','4','5','6','7','8','9','10',
        '11','12','binh-thanh','phu-nhuan','binh-tan',
        'go-vap','tan-binh','tan-phu','binh-chanh',
        'can-gio','cu-chi','hoc-mon','nha-be','thu-duc'
    ]

    # Danh sách tên quận hiển thị 
    DISTRICT_NAMES = [
        'Quận 1', 'Quận 2', 'Quận 3', 'Quận 4', 'Quận 5', 'Quận 6', 'Quận 7', 'Quận 8', 'Quận 9', 'Quận 10',
        'Quận 11', 'Quận 12', 'Bình Thạnh', 'Phú Nhuận', 'Bình Tân',
        'Gò Vấp', 'Tân Bình', 'Tân Phú', 'Bình Chánh',
        'Cần Giờ', 'Củ Chi', 'Hóc Môn', 'Nhà Bè', 'Thủ Đức'
    ]
    # Tạo map tên file -> tên quận hiển thị
    DISTRICT_MAP = dict(zip(DISTRICT_NAMES, DISTRICT_NAMES))

    def __init__(self, base_path: str, scraper: ChromeScraper, flattener: Flattener, file_lock: threading.Lock):
        # Thư mục gốc lưu dữ liệu
        self.base_path = base_path

        # Đối tượng ChromeScraper để crawl 1 URL
        self.scraper = scraper

        # Đối tượng flattener để xử lý JSON trả về
        self.flattener = flattener

        # Lock dùng để ghi file an toàn khi chạy đa luồng
        self.lock = file_lock

    def crawl_single_url(self, idx: int, row: pd.Series, file_name: str) -> bool:
        try:
            # Lấy id và URL từ dòng CSV
            id_val = row['id']
            url = row['url']

            # Convert tên file CSV -> tên quận hiển thị
            district = self.DISTRICT_MAP.get(file_name, "")

            # Đường dẫn folder lưu data crawl cho từng quận
            folder_bds_data_path = os.path.join(OUT_PATH, f"bds_{file_name}_data")

            # Tạo folder nếu chưa tồn
            os.makedirs(folder_bds_data_path, exist_ok=True)

            # Crawl dữ liệu thô từ trang
            data = self.scraper.scrape_specs(url, headless=True)

            # Làm gọn dữ liệu JSON trả về
            flat_data = self.flattener.flatten_listing(data)

            # Kiểm tra dữ liệu hợp lệ
            if not flat_data or not isinstance(flat_data, dict):
                print(f"⚠️ Không có dữ liệu hợp lệ từ {url}")
                return False

            # Gộp dữ liệu meta + dữ liệu đã xử lý
            full_data = {
                'id': id_val,
                'url': url,
                'district': district,
                **flat_data
            }

            # File JSON sẽ được lưu theo id
            full_data_path = os.path.join(folder_bds_data_path, f"{int(id_val)}.json")

            # Ghi file (có lock đảm bảo không đè nhau khi nhiều thread ghi)
            with self.lock:
                with open(full_data_path, 'w', encoding='utf-8') as f:
                    json.dump(full_data, f, indent=2, ensure_ascii=False, default=Utils.convert_np)

            print(f"✅ Đã ghi dữ liệu từ: {url}\n")
            return True
        except Exception as e:
            print(f"❌ Lỗi khi crawl index {idx}, URL {row.get('url', None)}: {e}")
            return False

In [6]:
# Quản lý ThreadPool, đọc CSV và phân phối nhiệm vụ cho worker
class CrawlerManager:

    def __init__(self, base_path: str = BASE_PATH, max_workers: int = MAX_WORKERS):
        # Thư mục gốc chứa file CSV và nơi lưu dữ liệu crawl
        self.base_path = base_path

        # Số luồng tối đa để chạy đa luồng
        self.max_workers = max_workers

        # Lock để tránh việc nhiều luồng ghi file cùng lúc gây lỗi
        self.lock = threading.Lock()

        # Chuẩn bị scraper Chrome (headless)
        self.scraper = ChromeScraper(headless=True)

        # Bộ xử lý dữ liệu sau khi crawl (làm gọn / chọn thông tin cần thiết)
        self.flattener = Flattener()

        # Worker phụ trách crawl từng URL riêng lẻ
        self.worker = CrawlerWorker(
            base_path=self.base_path,
            scraper=self.scraper,
            flattener=self.flattener,
            file_lock=self.lock
        )

    def run(self, file_name: str, index_range: Optional[Iterable[int]] = None) -> None:
        # Đường dẫn file CSV chứa danh sách URL
        file_csv_path = os.path.join(self.base_path, file_name + '_urls.csv')

        # Đọc danh sách URL vào DataFrame
        url_data = pd.read_csv(file_csv_path)
        max_index = len(url_data)

        # Nếu không chỉ định index → crawl toàn bộ
        if index_range is None:
            index_to_crawl = range(0, max_index)
        else:
            # Nếu user truyền range/list → dùng đúng phần đó
            index_to_crawl = index_range

        # Khởi tạo ThreadPoolExecutor để chạy đa luồng
        with ThreadPoolExecutor(max_workers=self.max_workers) as executor:
            futures = {}

            # Submit từng URL cho worker xử lý
            for idx in index_to_crawl:
                row = url_data.iloc[idx]
                future = executor.submit(self.worker.crawl_single_url, idx, row, file_name)
                futures[future] = idx

            # Theo dõi từng task hoàn thành
            for future in as_completed(futures):
                idx_done = futures[future]
                try:
                    # future.result() sẽ throw exception nếu task bị lỗi
                    future.result()
                except Exception as e:
                    print(f"❌ Lỗi khi crawl index {idx_done}: {e}")

        print(f"\n💾 Hoàn tất crawl, dữ liệu nằm trong: bds_{file_name}_data")


In [ ]:
if __name__ == "__main__":
    # Cấu hình / tham số chạy
    file_name = '8'
    MAX_CRAWL_PER_RUN = 100
    CRAWL_ALL = False

    manager = CrawlerManager(base_path=BASE_PATH, max_workers=MAX_WORKERS)

    # Đọc CSV để quyết định index nào sẽ crawl 
    file_csv_path = os.path.join(BASE_PATH, file_name + '_urls.csv')
    url_data = pd.read_csv(file_csv_path)
    max_index = len(url_data)


    index_to_crawl = range(0, 881)

    manager.run(file_name=file_name, index_range=index_to_crawl)


➡️ Đang mở trang: https://batdongsan.com.vn/ban-can-ho-chung-cu-duong-cao-lo-phuong-4-15-prj-topaz-elite/chinh-chu-ket-tien-ban-gap-3pn-moi-100-chua-qua-su-dung-pr44493828
➡️ Đang mở trang: https://batdongsan.com.vn/ban-shophouse-nha-pho-thuong-mai-duong-ben-binh-dong-phuong-14-4-prj-can-ho-d-aqua/ban-30-gia-5-2ty-an-thien-co-ban-co-wc-tro-ls-15-thang-thue-lai-24-thang-pr44411596
➡️ Đang mở trang: https://batdongsan.com.vn/ban-nha-biet-thu-lien-ke-duong-phu-dinh-phuong-16-1/-pho-1-tret-1-lung-2-lau-tai-tphcm-cach-vo-van-kiet-500m-chi-6ty390-lh-ngay-ms-oanh-pr44496343
⏳ Đang chờ tải thông tin cơ bản...
⏳ Đang chờ phần thông tin chi tiết...
⏳ Đang chờ tải thông tin cơ bản...
⏳ Đang chờ phần thông tin chi tiết...
⏳ Đang chờ tải thông tin cơ bản...
⏳ Đang chờ phần thông tin chi tiết...
✅ Tìm thấy 8 thông số kỹ thuật.
✅ Tìm thấy 3 thông số kỹ thuật.
✅ Tìm thấy 6 thông số kỹ thuật.
✅ Đã ghi dữ liệu từ: https://batdongsan.com.vn/ban-shophouse-nha-pho-thuong-mai-duong-ben-binh-dong-phuong-14-4